# Fine-Tune Qwen3:14B on Hugging Face Resume Dataset using Unsloth + QLoRA

This Google Colab notebook provides an end-to-end, error-free, GPU-aware pipeline to fine-tune **Qwen3:14B** (or `unsloth/Qwen2.5-14B-Instruct-bnb-4bit`) on the Hugging Face dataset `ganchengguang/resume_seven_class`.

### Key Highlights:
- **Zero Assumptions**: Automatically inspects dataset structure and detects resume text & category label columns.
- **Automatic GPU Scaling**: Detects GPU type (T4, L4, A100, V100) and scales sequence length, batch size, and precision dynamically.
- **Robust Data Cleaning**: Normalizes Unicode, removes nulls, empty resumes, and exact duplicates with audit logging.
- **PEFT QLoRA Optimization**: Uses Unsloth 4-bit FastLanguageModel for 2x faster training and 80% memory saving.
- **Production Inference & Export**: Interactive function for confidence scores, top-k candidates, and ZIP packaging.

## Step 1: Install Dependencies
Install latest compatible versions of Unsloth, Transformers, Datasets, TRL, PEFT, Accelerate, and BitsAndBytes.

In [ ]:
%%capture
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes sentencepiece huggingface_hub
!pip install datasets torch torchvision torchaudio scikit-learn matplotlib seaborn pandas numpy


## Step 2: GPU Detection & Memory Auto-Scaling
Detect GPU memory and select optimal max sequence length, batch size, gradient accumulation, and precision (fp16/bf16).

In [ ]:
import torch
import logging

print("=" * 60)
print("1. HARDWARE & ENVIRONMENT DETECTION")
print("=" * 60)

config = {
    "gpu_name": "CPU",
    "vram_gb": 0.0,
    "max_seq_length": 2048,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 2e-4,
    "fp16": True,
    "bf16": False,
    "lora_r": 16,
    "lora_alpha": 16,
}

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    config["gpu_name"] = device_name
    config["vram_gb"] = round(vram_gb, 2)
    print(f"Detected GPU: {device_name} | VRAM: {vram_gb:.2f} GB")
    
    if "A100" in device_name or "H100" in device_name or vram_gb >= 35:
        config.update({"max_seq_length": 4096, "per_device_train_batch_size": 4, "gradient_accumulation_steps": 2, "fp16": False, "bf16": True, "lora_r": 32, "lora_alpha": 32})
    elif "L4" in device_name or "V100" in device_name or vram_gb >= 20:
        config.update({"max_seq_length": 4096, "per_device_train_batch_size": 2, "gradient_accumulation_steps": 4, "fp16": not torch.cuda.is_bf16_supported(), "bf16": torch.cuda.is_bf16_supported()})
    else:
        config.update({"max_seq_length": 2048, "per_device_train_batch_size": 1, "gradient_accumulation_steps": 8, "fp16": True, "bf16": False})
else:
    print("WARNING: Running on CPU!")

print("Calculated Optimization Profile:", config)


## Step 3: Load Dataset & Comprehensive Inspection
Load dataset `ganchengguang/resume_seven_class` and inspect schema, row counts, null values, and duplicates.

In [ ]:
import pandas as pd
from datasets import load_dataset

dataset_name = "ganchengguang/resume_seven_class"
print(f"Loading dataset: {dataset_name}...")

try:
    ds = load_dataset(dataset_name)
    split = "train" if "train" in ds else list(ds.keys())[0]
    df = ds[split].to_pandas()
except Exception as e:
    print(f"Dataset load fallback required: {e}")
    url = f"https://huggingface.co/datasets/{dataset_name}/raw/main/resume.txt"
    df = pd.read_csv(url, sep="\n", header=None, names=["text"])

print("\n--- Dataset Inspection Summary ---")
print(f"Total Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"Null Counts:\n{df.isnull().sum()}")
print(f"Duplicates: {df.duplicated().sum()}")
print("\n--- First 5 Rows ---")
display(df.head(5))


## Step 4: Automatic Column Detection & Delimiter Parsing
Detect resume text and target label columns dynamically without assuming fixed column names. Auto-split tabbed data if present.

In [ ]:
resume_keywords = ["resume", "resume_text", "cv", "text", "content", "document", "user_resume"]
label_keywords = ["label", "category", "class", "role", "occupation", "target", "job_category"]

detected_resume_col = None
detected_label_col = None

cols_lower = {c: str(c).lower().strip() for c in df.columns}
for kw in resume_keywords:
    for orig, low in cols_lower.items():
        if kw in low:
            detected_resume_col = orig
            break
    if detected_resume_col: break

for kw in label_keywords:
    for orig, low in cols_lower.items():
        if kw in low:
            detected_label_col = orig
            break
    if detected_label_col: break

# Check single-column tab separation
if len(df.columns) == 1 or (detected_resume_col and detected_label_col is None):
    target_col = detected_resume_col if detected_resume_col else df.columns[0]
    samples = df[target_col].dropna().head(20).astype(str)
    if sum("\t" in s for s in samples) > 10:
        print(f"Splitting tab-separated data inside '{target_col}'...")
        split_df = df[target_col].astype(str).str.split("\t", n=1, expand=True)
        if split_df.shape[1] == 2:
            df["label"] = split_df[0].str.strip()
            df["resume_text"] = split_df[1].str.strip()
            detected_label_col, detected_resume_col = "label", "resume_text"

# Fallback Heuristics
if not detected_resume_col:
    str_cols = [c for c in df.columns if df[c].dtype in ["object", "string"]]
    if str_cols:
        lens = {c: df[c].dropna().astype(str).map(len).mean() for c in str_cols}
        detected_resume_col = max(lens, key=lens.get)

if not detected_label_col:
    other_cols = [c for c in df.columns if c != detected_resume_col]
    for c in other_cols:
        if 2 <= df[c].nunique() <= 50:
            detected_label_col = c
            break

if not detected_resume_col or not detected_label_col:
    raise ValueError(f"Could not detect columns from {list(df.columns)}. Diagnostic stopped.")

print(f"Detected Resume Text Column: '{detected_resume_col}'")
print(f"Detected Label Category Column: '{detected_label_col}'")


## Step 5: Dataset Cleaning & Unicode Sanitization
Filter nulls, whitespace, unicode anomalies, empty resumes, and exact duplicates. Generate cleaning audit report.

In [ ]:
import unicodedata
import re

orig_len = len(df)
df_clean = df.dropna(subset=[detected_resume_col, detected_label_col]).copy()

def clean_str(val):
    s = unicodedata.normalize("NFKD", str(val)).strip()
    return re.sub(r"[ \t]+", " ", s)

df_clean[detected_resume_col] = df_clean[detected_resume_col].apply(clean_str)
df_clean[detected_label_col] = df_clean[detected_label_col].apply(clean_str)

df_clean = df_clean[df_clean[detected_resume_col].str.len() > 10]
df_clean = df_clean[df_clean[detected_label_col].str.len() > 0]
df_clean = df_clean.drop_duplicates(subset=[detected_resume_col]).copy()

print("--- Cleaning Audit Report ---")
print(f"Original Samples: {orig_len}")
print(f"Cleaned Samples:  {len(df_clean)}")
print(f"Filtered Rows:    {orig_len - len(df_clean)}")
print("\n--- Top Categories ---")
print(df_clean[detected_label_col].value_counts().to_string())


## Step 6: Load Qwen3:14B using Unsloth & Configure QLoRA
Load 4-bit quantized model and initialize PEFT LoRA layers with optimal target modules.

In [ ]:
from unsloth import FastLanguageModel

model_name = "unsloth/Qwen2.5-14B-Instruct-bnb-4bit"
print(f"Loading model '{model_name}' in 4-bit with Unsloth...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=config["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=config["lora_r"],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=config["lora_alpha"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("Unsloth QLoRA setup completed successfully!")


## Step 7: Format Dataset into Qwen Instruction Format
Format samples into System-User-Assistant structure using `tokenizer.apply_chat_template`.

In [ ]:
from datasets import Dataset

prompt_inst = "Predict the most suitable job category for this candidate resume."

def format_entry(row):
    messages = [
        {"role": "system", "content": "You are an expert AI HR recruiter and job classification system."},
        {"role": "user", "content": f"{prompt_inst}\n\nCandidate Resume:\n{row[detected_resume_col]}"},
        {"role": "assistant", "content": str(row[detected_label_col])}
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

formatted_data = [format_entry(row) for _, row in df_clean.iterrows()]
hf_dataset = Dataset.from_pandas(pd.DataFrame(formatted_data))
print(f"Formatted {len(hf_dataset)} instruction samples!")


## Step 8: Configure Trainer & Execute Fine-Tuning
Set up TRL SFTTrainer with cosine LR schedule, warmup, weight decay, and auto-checkpointing.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

ds_split = hf_dataset.train_test_split(test_size=0.1, seed=42)

training_args = TrainingArguments(
    per_device_train_batch_size=config["per_device_train_batch_size"],
    gradient_accumulation_steps=config["gradient_accumulation_steps"],
    warmup_ratio=0.05,
    max_steps=100, # Set to desired training steps or epochs
    learning_rate=config["learning_rate"],
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    output_dir="./qwen3_outputs",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds_split["train"],
    eval_dataset=ds_split["test"],
    dataset_text_field="text",
    max_seq_length=config["max_seq_length"],
    packing=False,
    args=training_args,
)

print("Starting fine-tuning...")
trainer_stats = trainer.train()
print("Training Complete!")


## Step 9: Production Inference Pipeline
Construct inference function returning predicted category, confidence score, and top-5 candidates.

In [ ]:
FastLanguageModel.for_inference(model)

def predict_resume(resume_text: str, top_k: int = 5):
    messages = [
        {"role": "system", "content": "You are an expert AI HR recruiter and job classification system."},
        {"role": "user", "content": f"{prompt_inst}\n\nCandidate Resume:\n{resume_text}"}
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(input_ids=inputs, max_new_tokens=64, temperature=0.1, return_dict_in_generate=True, output_scores=True)
        
    gen_tokens = outputs.sequences[0][inputs.shape[1]:]
    pred_text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    
    if outputs.scores:
        probs = torch.softmax(outputs.scores[0][0], dim=-1)
        top_p, top_i = torch.topk(probs, k=top_k)
        top_cats = [(tokenizer.decode([idx.item()]).strip(), round(prob.item(), 4)) for prob, idx in zip(top_p, top_i)]
        conf = round(top_p[0].item(), 4)
    else:
        conf, top_cats = 1.0, [(pred_text, 1.0)]
        
    return pred_text, conf, top_cats

# Test Sample
test_resume = "Senior Data Scientist with 6 years experience in Python, PyTorch, LLMs, NLP, and SQL."
cat, conf, top5 = predict_resume(test_resume)
print(f"Predicted Category: {cat} (Confidence: {conf})")
print(f"Top-5 Candidates: {top5}")


## Step 10: Evaluation & Metrics Visualization
Calculate Accuracy, Precision, Recall, F1 Score, and plot Confusion Matrix.

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

val_sample = df_clean.sample(min(30, len(df_clean)), random_state=42)
y_true = val_sample[detected_label_col].tolist()
y_pred = [predict_resume(text)[0] for text in val_sample[detected_resume_col]]

acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)

print(f"Validation Accuracy:  {acc:.4f}")
print(f"Validation Precision: {prec:.4f}")
print(f"Validation Recall:    {rec:.4f}")
print(f"Validation F1 Score:  {f1:.4f}")

labels_list = sorted(list(set(y_true + y_pred)))
cm = confusion_matrix(y_true, y_pred, labels=labels_list)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels_list, yticklabels=labels_list)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()


## Step 11: Save Adapters & Package ZIP
Save fine-tuned QLoRA weights and download zip archive.

In [ ]:
import zipfile
import os

out_dir = "./qwen3_resume_lora"
model.save_pretrained(out_dir)
tokenizer.save_pretrained(out_dir)

zip_name = "qwen3_resume_fine_tuned.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(out_dir):
        for file in files:
            path = os.path.join(root, file)
            zipf.write(path, os.path.relpath(path, out_dir))

print(f"Saved LoRA adapter and created package: '{zip_name}'")
